# Task 3: Forecast Future Market Trends

**GMF Investments — TSLA 6-Month Forward Forecast**

**Author:** Sosina Ayele

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pickle, os, warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
print('Libraries loaded!')

## 1. Load Best Model & Full Data

In [ ]:
# Load prices
csv_path = '../data/processed/prices.csv'
if os.path.exists(csv_path):
    prices = pd.read_csv(csv_path, index_col=0, parse_dates=True)
else:
    raw = yf.download(['TSLA','BND','SPY'], start='2015-01-01', end='2026-06-30', auto_adjust=True)
    prices = raw['Close'].ffill().bfill()

tsla = prices['TSLA'].dropna()
train = tsla[tsla.index < '2025-01-01']
test  = tsla[tsla.index >= '2025-01-01']

WINDOW = 60
FORECAST_DAYS = 126  # ~6 months of trading days

# Load scaler and model
if os.path.exists('../models/scaler.pkl'):
    with open('../models/scaler.pkl','rb') as f:
        scaler = pickle.load(f)
else:
    scaler = MinMaxScaler(feature_range=(0,1))
    scaler.fit(train.values.reshape(-1,1))

if os.path.exists('../models/lstm_tsla.keras'):
    model = tf.keras.models.load_model('../models/lstm_tsla.keras')
    print('LSTM model loaded!')
else:
    print('No saved model found — please run Task 2 first')

print(f'Last known price: ${tsla.iloc[-1]:.2f} on {tsla.index[-1].date()}')

## 2. Generate 6-Month Future Forecast

In [ ]:
# Multi-step forecast: feed predictions back iteratively
all_scaled = scaler.transform(tsla.values.reshape(-1,1))
last_sequence = all_scaled[-WINDOW:].flatten().tolist()

future_preds = []
for _ in range(FORECAST_DAYS):
    seq = np.array(last_sequence[-WINDOW:]).reshape(1, WINDOW, 1)
    pred = model.predict(seq, verbose=0)[0][0]
    future_preds.append(pred)
    last_sequence.append(pred)

# Inverse transform
future_prices = scaler.inverse_transform(
    np.array(future_preds).reshape(-1,1)).flatten()

# Create future date index (business days)
last_date = tsla.index[-1]
future_dates = pd.bdate_range(start=last_date + pd.Timedelta(days=1),
                               periods=FORECAST_DAYS)
future_series = pd.Series(future_prices, index=future_dates)

print(f'Forecast period: {future_dates[0].date()} to {future_dates[-1].date()}')
print(f'Current price: ${tsla.iloc[-1]:.2f}')
print(f'Forecast end:  ${future_prices[-1]:.2f}')
print(f'Expected change: {(future_prices[-1]/tsla.iloc[-1]-1)*100:+.1f}%')

## 3. Visualize Forecast with Confidence Intervals

In [ ]:
# Build confidence intervals based on historical volatility
daily_vol = tsla.pct_change().std()
conf_95 = []
conf_80 = []
for i in range(1, FORECAST_DAYS+1):
    ci_95 = future_prices[i-1] * daily_vol * np.sqrt(i) * 1.96
    ci_80 = future_prices[i-1] * daily_vol * np.sqrt(i) * 1.28
    conf_95.append(ci_95)
    conf_80.append(ci_80)

lower_95 = future_prices - np.array(conf_95)
upper_95 = future_prices + np.array(conf_95)
lower_80 = future_prices - np.array(conf_80)
upper_80 = future_prices + np.array(conf_80)

# Plot
fig, ax = plt.subplots(figsize=(16, 7))

# Historical
ax.plot(tsla[-200:].index, tsla[-200:], color='steelblue',
        label='Historical TSLA', linewidth=1.5)
# Test period
ax.plot(test.index, test, color='black', linewidth=1.5, label='Actual (test)')
# Forecast
ax.plot(future_dates, future_prices, color='#e74c3c',
        linewidth=2, linestyle='--', label='LSTM Forecast')
# CIs
ax.fill_between(future_dates, lower_95, upper_95,
                alpha=0.15, color='#e74c3c', label='95% CI')
ax.fill_between(future_dates, lower_80, upper_80,
                alpha=0.25, color='#e74c3c', label='80% CI')

ax.axvline(last_date, color='grey', linestyle=':', alpha=0.8, label='Forecast start')
ax.set_title('TSLA 6-Month Forward Forecast with Confidence Intervals',
             fontweight='bold', fontsize=14)
ax.set_ylabel('Price (USD)', fontsize=12)
ax.legend(loc='upper left', fontsize=10)
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('future_forecast.png', dpi=150, bbox_inches='tight')
plt.show()
print('Forecast plot saved!')

## 4. Trend Analysis & Opportunities/Risks

In [ ]:
# Trend decomposition
trend = pd.Series(future_prices, index=future_dates)
monthly = trend.resample('ME').last()

print('=== MONTHLY FORECAST SUMMARY ===')
for date, price in monthly.items():
    chg = (price/tsla.iloc[-1]-1)*100
    print(f'{date.strftime("%b %Y")}: ${price:.2f} ({chg:+.1f}% from today)')

print(f'\n=== CONFIDENCE INTERVAL WIDTH ===')
for i, (d, l, u) in enumerate(zip(future_dates[::21], lower_95[::21], upper_95[::21])):
    width = u - l
    print(f'Month {i+1} ({d.strftime("%b %Y")}): ${l:.0f} - ${u:.0f} (width: ${width:.0f})')

print(f'\n=== OPPORTUNITIES & RISKS ===')
peak_idx = np.argmax(future_prices)
trough_idx = np.argmin(future_prices)
print(f'Forecast peak: ${future_prices[peak_idx]:.2f} on {future_dates[peak_idx].date()}')
print(f'Forecast trough: ${future_prices[trough_idx]:.2f} on {future_dates[trough_idx].date()}')
print(f'\nOPPORTUNITY: If forecast is accurate, entry near trough represents '
      f'{(future_prices[peak_idx]/future_prices[trough_idx]-1)*100:.1f}% gain potential')
print(f'RISK: 95% CI reaches ${lower_95[-1]:.0f}-${upper_95[-1]:.0f} by month 6 '
      f'— widening uncertainty limits long-term conviction')

future_series.to_csv('../data/processed/tsla_forecast.csv', header=['forecast'])
print('\nForecast saved!')